# ♻️ Smart Waste Classification using ResNet50

## Project Overview

This project develops a deep learning-based waste classification system capable of classifying waste images into six categories:

- Plastic Bottle
- Carton
- Metal Packaging
- Household Waste
- Paper & Cardboard
- Glass

The project uses a pretrained ResNet50 model with transfer learning, fine-tuning, data augmentation, and learning-rate scheduling.

After comparing different training strategies, the best-performing model achieved a test accuracy of **74.52%** and was deployed as an interactive web application using Streamlit.

## Objectives

- Build an image classification model for six waste categories.
- Apply transfer learning using a pretrained ResNet50 architecture.
- Experiment with data augmentation and fine-tuning.
- Use learning-rate scheduling to study its effect on training.
- Compare different training strategies objectively.
- Save and deploy the best-performing model using Streamlit.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

In [ ]:
# Explore the dataset directory

import os

for dirname, dirnames, filenames in os.walk("/kaggle/input"):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/manonstr
/kaggle/input/datasets/manonstr/tipe-webscraping
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement/Bouteille_plastique
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement/Emballage_metallique
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement/Papier_Carton
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement/Brique_en_carton
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement/Verre
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/entrainement/Ordure_m�nag�re
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test/Bouteille_plastique
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test/Emballage_metallique
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test/Papier_Carton
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test/Brique_en_carton
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test/Verre
/kaggle/input/datasets/manonstr/tipe-webscraping/1024+/test/Ordure_m�nag�re

## 📂 Dataset

The dataset contains waste images organized into six categories.

The original dataset class labels are:

1. `Bouteille_plastique`
2. `Brique_en_carton`
3. `Emballage_metallique`
4. `Ordure_ménagère`
5. `Papier_Carton`
6. `Verre`

For the deployed application, these labels are displayed in English as:

| Original Label | English Label |
|---|---|
| Bouteille_plastique | Plastic Bottle |
| Brique_en_carton | Carton |
| Emballage_metallique | Metal Packaging |
| Ordure_ménagère | Household Waste |
| Papier_Carton | Paper & Cardboard |
| Verre | Glass |

## 🔄 Data Preprocessing and Augmentation

All images are resized to `224 × 224` pixels because this is the standard input size used by ResNet50.

Training-time data augmentation is applied to improve generalization by introducing realistic variations in the training images.

The preprocessing pipeline includes:

- Resize to 224 × 224
- Random horizontal flipping
- Random rotation
- Color jitter
- Conversion to tensor
- ImageNet normalization

Validation and test images are not randomly augmented because evaluation should be performed on unchanged images.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
## Load Dataset and DataLoaders
train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=test_path,
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Training Images:", len(train_dataset))
print("Test Images:", len(test_dataset))
print("Classes:", train_dataset.classes)

In [ ]:
## Visualize Sample Images
images, labels = next(iter(train_loader))

fig = plt.figure(figsize=(12, 8))

for i in range(min(6, len(images))):
    image = images[i].permute(1, 2, 0).numpy()

    # Reverse ImageNet normalization for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    image = std * image + mean
    image = np.clip(image, 0, 1)

    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.title(train_dataset.classes[labels[i].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 🧠 Stage 1: Transfer Learning with ResNet50

ResNet50 is a convolutional neural network pretrained on the ImageNet dataset.

Instead of training the entire network from scratch, transfer learning reuses the pretrained feature extractor.

During this stage:

1. A pretrained ResNet50 model is loaded.
2. All pretrained layers are frozen.
3. The final fully connected classification layer is replaced.
4. Only the new classifier is trained for the six waste categories.

This approach is computationally efficient and particularly useful when working with a relatively limited dataset.

In [ ]:
model = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# Freeze all pretrained parameters
for param in model.parameters():
    param.requires_grad = False

# Replace final classification layer
num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    len(train_dataset.classes)
)

model = model.to(device)

print(model.fc)

In [ ]:
## Loss and Optimizer
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

In [ ]:
def train_model(model, train_loader, criterion, optimizer, num_epochs):

    training_losses = []

    for epoch in range(num_epochs):

        model.train()

        running_loss = 0.0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        training_losses.append(epoch_loss)

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"Loss: {epoch_loss:.4f}"
        )

    return training_losses

In [ ]:
## Train Transfer-Learning Model
num_epochs = 10

transfer_losses = train_model(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=num_epochs
)

In [ ]:
## Evaluation Function
def evaluate_model(model, data_loader):

    model.eval()

    correct = 0
    total = 0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in data_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

            all_predictions.extend(
                predicted.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    accuracy = 100 * correct / total

    print("Correct Predictions :", correct)
    print("Total Images        :", total)
    print(f"Test Accuracy       : {accuracy:.2f}%")

    return accuracy, all_labels, all_predictions

In [ ]:
## Evaluate Transfer Learning
transfer_accuracy, true_labels, predictions = evaluate_model(
    model,
    test_loader
)

In [ ]:
# Save the transfer-learning model before fine-tuning modifies its weights

torch.save(
    model.state_dict(),
    "/kaggle/working/best_transfer_model.pth"
)

print(
    f"Transfer-learning model saved successfully "
    f"with accuracy: {transfer_accuracy:.2f}%"
)

## 🔧 Stage 2: Fine-Tuning

After transfer learning, fine-tuning was performed to determine whether allowing deeper layers of ResNet50 to adapt to the waste dataset would improve generalization.

The final residual block (`layer4`) was unfrozen while the earlier layers remained frozen.

A smaller learning rate was used because pretrained weights should be updated carefully.

In this experiment, fine-tuning did not outperform the original transfer-learning model. This is a valid experimental result and suggests that additional weight updates may have caused overfitting or disrupted useful pretrained features.

In [ ]:
# Freeze all parameters first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze the final residual block
for param in model.layer4.parameters():
    param.requires_grad = True

# Keep classifier trainable
for param in model.fc.parameters():
    param.requires_grad = True


trainable_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable Parameters:", trainable_count)

In [ ]:
optimizer = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=0.0001
)

In [ ]:
fine_tuning_losses = train_model(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=10
)

In [ ]:
fine_tuning_accuracy, _, _ = evaluate_model(
    model,
    test_loader
)

## 📉 Stage 3: Learning-Rate Scheduling

A learning-rate scheduler was tested to gradually reduce the learning rate during training.

The goal was to make larger updates early in training and smaller, more precise updates later.

However, in this experiment, the scheduler did not improve test accuracy beyond the original transfer-learning model.

In [ ]:
optimizer = optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=0.0001
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

In [ ]:
scheduler_losses = []

for epoch in range(10):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    scheduler_losses.append(epoch_loss)

    scheduler.step()

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/10] "
        f"Loss: {epoch_loss:.4f}"
    )

    print(
        f"Learning Rate: {current_lr}"
    )

In [ ]:
# Evaluate model 

scheduler_accuracy, _, _ = evaluate_model(
    model,
    test_loader
)

print(
    f"Scheduler Test Accuracy: "
    f"{scheduler_accuracy:.2f}%"
)

## 📊 Comparison of Training Strategies

Three training strategies were evaluated:

| Experiment | Test Accuracy |
|---|---:|
| Transfer Learning | **74.52%** |
| Fine-Tuning | 71.34% |
| Fine-Tuning + LR Scheduler | 71.34% |

The transfer-learning model achieved the highest test accuracy and was therefore selected as the final model for deployment.

An important observation from this experiment is that more complex training strategies do not always guarantee better generalization. Model selection should be based on validation or test performance rather than training loss alone.

In [ ]:
methods = [
    "Transfer Learning",
    "Fine-Tuning",
    "Fine-Tuning + Scheduler"
]

accuracies = [
    transfer_accuracy,
    fine_tuning_accuracy,
    scheduler_accuracy
]


plt.figure(figsize=(10, 6))

bars = plt.bar(
    methods,
    accuracies
)

plt.title("Comparison of Training Strategies")
plt.xlabel("Training Strategy")
plt.ylabel("Test Accuracy (%)")
plt.ylim(0, 100)

for bar, accuracy in zip(bars, accuracies):

    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{accuracy:.2f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    range(1, len(transfer_losses) + 1),
    transfer_losses,
    marker="o",
    label="Transfer Learning"
)

plt.plot(
    range(1, len(fine_tuning_losses) + 1),
    fine_tuning_losses,
    marker="o",
    label="Fine-Tuning"
)

plt.title("Training Loss Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 🌐 Model Deployment

The best-performing model was integrated into a Streamlit web application.

The deployment pipeline consists of:

1. User uploads an image.
2. The image is converted to RGB.
3. It is resized to 224 × 224 pixels.
4. ImageNet normalization is applied.
5. The image is passed through the trained ResNet50 model.
6. Softmax converts the output logits into probabilities.
7. The predicted waste category and confidence score are displayed.

A confidence threshold of 65% is used to warn users when the model is uncertain about a prediction.

## ⚠️ Limitations

The model is a closed-set image classifier trained exclusively on six waste categories.

Therefore, if an unrelated image such as an animal, flower, person, or other unsupported object is uploaded, the model is still forced to select one of the six known classes.

A confidence threshold can help identify some uncertain predictions, but it does not guarantee reliable out-of-distribution detection because neural networks can sometimes produce high confidence scores for unfamiliar inputs.

Future work could include:

- An explicit `Unknown / Other` class
- Out-of-distribution detection
- A larger and more diverse dataset
- Additional waste categories
- Object detection for images containing multiple waste objects

## 🎯 Conclusion

This project developed an end-to-end deep learning system for classifying waste images into six categories using ResNet50 and PyTorch.

Three training strategies were explored:

- Transfer Learning
- Fine-Tuning
- Learning-Rate Scheduling

The transfer-learning model achieved the best test accuracy of **74.52%** and was selected for deployment.

An important experimental finding was that additional fine-tuning did not improve test performance in this case. This demonstrates that more complex training strategies do not necessarily guarantee better generalization, and model selection should be based on objective evaluation results.

The final model was integrated into a Streamlit web application that allows users to upload waste images and receive predicted categories with confidence scores.